# Capitolo 3 — Esperimenti guidati: ottimizzatori, inizializzazione, tasso, batch (§ 3.9)
Quattro confronti sul MLP di MNIST. Ogni sezione dura da uno a tre minuti su un portatile.

In [ ]:
import sys; sys.path.insert(0, "..")
from utils import fissa_seme
import dati
import numpy as np
import matplotlib.pyplot as plt

import torch
from torch import nn
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
fissa_seme(42)

def mlp():
    return nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.ReLU(), nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 10))

perdita_fn = nn.CrossEntropyLoss()

def valuta(modello, dl):
    modello.eval(); corretti, totale, perdita_tot = 0, 0, 0.0
    with torch.no_grad():
        for xb, yb in dl:
            uscita = modello(xb)
            perdita_tot += perdita_fn(uscita, yb).item() * len(yb)
            corretti += (uscita.argmax(dim=1) == yb).sum().item(); totale += len(yb)
    return perdita_tot / totale, corretti / totale

def addestra(modello, ottimizzatore, train_ds, test_dl, epoche=5, batch_size=64, stampa=True):
    dl = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    storia = {"train": [], "test": [], "acc": [], "passi": []}
    for epoca in range(epoche):
        modello.train(); somma, n = 0.0, 0
        for i, (xb, yb) in enumerate(dl):
            perdita = perdita_fn(modello(xb), yb)
            ottimizzatore.zero_grad(); perdita.backward(); ottimizzatore.step()
            somma += perdita.item() * len(yb); n += len(yb)
            if i % 10 == 0: storia["passi"].append(perdita.item())
        pt, at = valuta(modello, test_dl)
        storia["train"].append(somma / n); storia["test"].append(pt); storia["acc"].append(at)
        if stampa: print(f"Epoca {epoca+1}: perdita train {somma/n:.4f} | perdita test {pt:.4f} | accuratezza test {at:.2%}")
    return storia

trasforma = transforms.ToTensor()
train_ds = datasets.MNIST("../data", train=True,  download=True, transform=trasforma)
test_ds  = datasets.MNIST("../data", train=False, download=True, transform=trasforma)
test_dl = DataLoader(test_ds, batch_size=1000)

def grafico(risultati, titolo, passo=10):
    fig, ax = plt.subplots(1, 2, figsize=(10, 3.4))
    for nome, st in risultati.items():
        p = np.array(st["passi"]); ax[0].plot(np.arange(len(p) - 4) * passo, np.convolve(p, np.ones(5) / 5, "valid"), lw=1, label=nome)
        ax[1].plot(range(1, len(st["acc"]) + 1), np.array(st["acc"]) * 100, "-o", label=nome)
    ax[0].set_xlabel("passi"); ax[0].set_ylabel("perdita sul batch"); ax[0].set_ylim(0, 2.5); ax[0].legend(fontsize=7); ax[0].grid(alpha=0.3); ax[0].set_title(titolo)
    ax[1].set_xlabel("epoca"); ax[1].set_ylabel("accuratezza test (%)"); ax[1].legend(fontsize=7); ax[1].grid(alpha=0.3)
    plt.show()

## Ottimizzatori

In [ ]:
risultati = {}
for nome, mk in [("SGD lr=0.01", lambda p: torch.optim.SGD(p, lr=0.01)), ("SGD lr=0.1", lambda p: torch.optim.SGD(p, lr=0.1)),
                 ("SGD+momentum 0.9, lr=0.01", lambda p: torch.optim.SGD(p, lr=0.01, momentum=0.9)), ("Adam lr=1e-3", lambda p: torch.optim.Adam(p, lr=1e-3))]:
    fissa_seme(42); m = mlp(); print(nome)
    risultati[nome] = addestra(m, mk(m.parameters()), train_ds, test_dl, epoche=5, stampa=False)
    print("  accuratezza per epoca:", [f"{a:.1%}" for a in risultati[nome]["acc"]])
grafico(risultati, "Ottimizzatori")

## Inizializzazione (SGD lr = 0.1, 3 epoche)

In [ ]:
risultati = {}
for nome, fn_ini in [("default (Kaiming)", None), ("tutti zero", lambda w: nn.init.zeros_(w)), ("normale std 1", lambda w: nn.init.normal_(w, 0, 1.0)), ("normale std 0.01", lambda w: nn.init.normal_(w, 0, 0.01))]:
    fissa_seme(42); m = mlp()
    if fn_ini is not None:
        for strato in m:
            if isinstance(strato, nn.Linear): fn_ini(strato.weight); nn.init.zeros_(strato.bias)
    risultati[nome] = addestra(m, torch.optim.SGD(m.parameters(), lr=0.1), train_ds, test_dl, epoche=3, stampa=False)
    print(nome, [f"{a:.1%}" for a in risultati[nome]["acc"]])
grafico(risultati, "Inizializzazione")

## Tasso di apprendimento (Adam, 3 epoche)

In [ ]:
risultati = {}
for lr in (1e-4, 1e-3, 1e-2, 1e-1):
    fissa_seme(42); m = mlp()
    risultati[f"lr = {lr}"] = addestra(m, torch.optim.Adam(m.parameters(), lr=lr), train_ds, test_dl, epoche=3, stampa=False)
    print(lr, [f"{a:.1%}" for a in risultati[f"lr = {lr}"]["acc"]])
grafico(risultati, "Tasso di apprendimento")

## Dimensione del batch (Adam, 3 epoche) — misura anche il tempo per epoca

In [ ]:
import time
for bs in (16, 64, 256, 4096):
    fissa_seme(42); m = mlp(); t0 = time.time()
    st = addestra(m, torch.optim.Adam(m.parameters(), lr=1e-3), train_ds, test_dl, epoche=3, batch_size=bs, stampa=False)
    print(f"batch {bs:5d}: {60000 // bs:5d} passi/epoca | acc {[f'{a:.1%}' for a in st['acc']]} | {(time.time() - t0) / 3:.1f} s/epoca")

## Esperimento libero: senza ReLU la profondità non serve

In [ ]:
fissa_seme(42)
senza_relu = nn.Sequential(nn.Flatten(), nn.Linear(784, 128), nn.Linear(128, 64), nn.Linear(64, 10))
st = addestra(senza_relu, torch.optim.Adam(senza_relu.parameters(), lr=1e-3), train_ds, test_dl, epoche=5, stampa=False)
print("Senza ReLU, dopo 5 epoche:", f"{st['acc'][-1]:.2%}")